In [ ]:
import requests
import random
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from collections import defaultdict
from conjugate_normal import conjugate_normal

In [ ]:
def get_score_history(user_id, current_timestamp, df_sorted):
    """Get actual score history for this user up to this point"""
    # Get all previous submissions for this user before current timestamp
    user_previous = df_sorted[
        (df_sorted['user_id'] == user_id) & 
        (df_sorted['created_at'] < current_timestamp)
    ]
    
    # Return their content scores as the score history
    if len(user_previous) > 0:
        return user_previous['content_score'].tolist()
    else:
        return []  # First submission for this user

def call_summary_api(summary_text, user_id, timestamp, df_sorted):
    url = "https://itell-api.learlab.vanderbilt.edu/score/summary"

    # Get actual score history for this user
    score_history = get_score_history(user_id, timestamp, df_sorted)

    payload = {
        "page_slug": "test-page",
        "class_id": "Tobasum-Testing-Joyner-Data",
        "score_history": score_history,
        "summary": summary_text
    }
    headers = {
        "Content-Type": "application/json",
        "API-Key": ""
    }

    response = requests.request("POST", url, json=payload, headers=headers)
    response_dict = response.json()
    response_dict['score_history'] = payload['score_history']
    return response_dict

In [ ]:
# Load the data from server
df = pd.read_csv("summaries.csv")

# Display basic info about the data
print(f"Total rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
print(f"Unique users: {df['user_id'].nunique()}")
print(f"Date range: {df['created_at'].min()} to {df['created_at'].max()}")

# Show sample data
display(df.head())

# Prepare data for API calls
required_cols = ['user_id', 'text', 'created_at'] 
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print(f"Missing columns: {missing_cols}")
    print("Available columns: {list(df.columns)}")
else:
    print("All required columns present")
    
# Sort by timestamp
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.sort_values('created_at')
print(f"Rows after sorting: {len(df)}")

Total rows: 994
Columns: ['id', 'text', 'condition', 'user_id', 'page_slug', 'is_passed', 'containment_score', 'similarity_score', 'content_score', 'content_threshold', 'is_excellent', 'created_at', 'updated_at']
Unique users: 127
Date range: 2025-06-02T15:40:05.637001+00:00 to 2025-07-21T19:10:20.558299+00:00


,id,text,condition,user_id,page_slug,is_passed,containment_score,similarity_score,content_score,content_threshold,is_excellent,created_at,updated_at
0,60,Thi spage talks about how contril stuctues are...,random_reread,226ajlpobyjbksx534h6rrg2jq,3-1-control-structures,True,0.0000,0.736030,0.170597,0.004953,True,2025-06-02T15:40:05.637001+00:00,2025-06-02T15:40:05.637001+00:00
1,61,"In this chapter of the unit 3 lessons, the art...",random_reread,226ajlpobyjbksx534h6rrg2jq,3-2-conditionals,True,0.0000,0.520184,0.077714,0.004953,True,2025-06-02T17:31:34.022806+00:00,2025-06-02T17:31:34.022806+00:00
2,62,"In this section of the lesson for unit 3, the ...",random_reread,226ajlpobyjbksx534h6rrg2jq,3-3-loops,True,0.0000,0.779228,0.349814,0.004953,True,2025-06-02T17:36:51.540634+00:00,2025-06-02T17:36:51.540634+00:00
3,63,"In this chapter of unit 3, the article talks a...",random_reread,226ajlpobyjbksx534h6rrg2jq,3-4-functions,True,0.0000,0.758652,0.611322,0.004953,True,2025-06-02T17:41:39.63132+00:00,2025-06-02T17:41:39.63132+00:00
4,64,"In this article for unit 3, the article talks ...",random_reread,226ajlpobyjbksx534h6rrg2jq,3-5-error-handling,True,0.1053,0.785503,0.431579,0.004953,True,2025-06-02T17:50:59.201884+00:00,2025-06-02T17:50:59.201884+00:00


All required columns present
Rows after sorting: 994


In [ ]:
print("Making API calls...")

all_responses = []
for i, row in enumerate(df.itertuples()):
    print(f"Processing summary {i+1}/{len(df)} for user {row.user_id}")
    
    summary_text = row.text
    user_id = row.user_id
    timestamp = row.created_at
    
    # Call API with actual score history for this user
    response = call_summary_api(summary_text, user_id, timestamp, df)
    
    if response:
        all_responses.append(response)
        print(f"  - Score history length: {len(response['score_history'])}")
    
    # Small delay
    import time
    time.sleep(0.1)

print(f"Completed {len(all_responses)} API calls")

# Save to JSONL file
with open("real_summary_scores.jsonl", "w") as f:
    for response in all_responses:
        f.write(json.dumps(response) + "\n")

print("Saved API responses to real_summary_scores.jsonl")

Initial volume threshold: 0.3410
Starting API simulation with 994 submissions...


In [ ]:
# Load and process the API responses
df1 = pd.read_json("real_summary_scores.jsonl", lines=True)
df2 = pd.json_normalize(df1["metrics"], sep="_")
df = pd.concat([df1, df2], axis=1)
df = df.dropna(subset=["content_score"])

print(f"Processed {len(df)} API responses")
display(df.sample(1))

In [ ]:
mu_global = 0.8
k_global = 15
alpha_global = 4
beta_global = 1

k_volume = 3

volumes = {}
volume_updates = []
submission_threshold = []

volume = conjugate_normal(mu_global, k_global, alpha_global, beta_global, percentile=0.2)
start_threshold = volume.threshold

student_score_history = defaultdict(list)

for i, row in enumerate(df.itertuples()):
    # Create temporary volume prior from volume prior information
    student_dist = conjugate_normal(volume.mu, k_volume, volume.alpha, volume.beta, percentile=0.2)

    # Update temporary volume prior with score history
    if row.score_history:
        student_dist.update(row.score_history)
        # Record threshold
        submission_threshold.append(student_dist.threshold)
    else:
        submission_threshold.append(volume.threshold)

    volume.update([row.content_score])
    volume_updates.append((volume.mu, volume.sigma, volume.threshold))

    
df["bayesian_threshold"] = submission_threshold


In [ ]:
sns.set_style("whitegrid")
fig, ax = plt.subplots(1, figsize=(16, 4))

pass_rates = []

# Calculate and plot rolling average (10 submissions)
rolling_avg = df['content_score'].rolling(window=10, min_periods=1).mean()
sns.lineplot(ax=ax, x=df.index, y=rolling_avg, 
             label='Content Score Rolling Average (10 Submissions)', legend=False,
             color='blue', linewidth=1, alpha=0.6)

# Draw Observed Threshold (as the student experienced it)
sns.lineplot(ax=ax, data=df, x=df.index, y='content_threshold', 
             label='Observed Threshold in Production', legend=False,
             color='green', linewidth=1, alpha=0.6, linestyle='dashed')

# Draw Volume Prior
# updates_array = np.array(volume_updates)
# sns.lineplot(ax=ax, x=df.index, y=updates_array[:,2], 
#              label='Simulated Bayesian Volume Prior Threshold', legend=False,
#              color='purple', linewidth=1, alpha=0.6, linestyle='dashed')

# Draw Bayesian Threshold
sns.lineplot(ax=ax, data=df, x=df.index, y='bayesian_threshold', 
             label='Simulated Bayesian Personalized Threshold', legend=False,
             color='orange', linewidth=1, alpha=0.6, linestyle='dashed')

# Add points for rows with non-empty score_history
mask = df['score_history'].apply(lambda x: isinstance(x, list) and len(x) > 0)
filtered_df = df[mask]

if len(filtered_df) > 0:
    ax.scatter(filtered_df.index, filtered_df['content_score'], 
               color='red', s=30, alpha=0.7, zorder=5,
               label='Non-empty Score History')

# Manually set the threshold for the first window_size samples
# Since we don't have enough data to calculate a fair rolling average
window_size=10
rolling_20th = df["content_score"].rolling(window=window_size, min_periods=1, closed='left').quantile(0.2)
rolling_20th.iloc[:window_size] = 0.0

# Print Pass Rates
pass_rates.append({
    "Observed": (df["content_score"] > df["content_threshold"]).astype(int).mean(),
    "Simulated Bayesian Personalized": (df["content_score"] > df["bayesian_threshold"]).astype(int).mean(),
    # "Simulated Bayesian Volume Prior Only": (df["content_score"] > updates_array[:,2]).astype(int).mean(),
    "20th Percentile": (df["content_score"] > rolling_20th).astype(int).mean(),
})

# Format x-axis to show only month-year
plt.gca().xaxis.set_major_formatter(mpl.dates.DateFormatter('%b'))
plt.gca().xaxis.set_major_locator(mpl.dates.MonthLocator(interval=3))

# Rotate x-axis labels
plt.xticks(rotation=45)

# Show Pass Rates
display(pd.DataFrame(pass_rates).round(2))

# Add legend
labels_handles = {}
for ax in fig.axes:
    for handle, label in zip(*ax.get_legend_handles_labels()):
        labels_handles[label] = handle

fig.legend(labels_handles.values(), labels_handles.keys(), loc='upper right', bbox_transform=fig.transFigure, ncol=1)

plt.title('Content Score Thresholds', pad=20)
plt.xlabel('Creation Date')
plt.ylabel('Content Score')
plt.tight_layout()
    
plt.show();
